In [0]:
USE CATALOG tibia_analytics;
USE SCHEMA silver;
SET TIME ZONE 'UTC';

In [0]:
CREATE OR REPLACE TABLE tibia_analytics.silver.tmp_characters_history_enriched AS
WITH filtered_state AS (
  SELECT character_id,
         name,
         valid_from,
         valid_to
    FROM tibia_analytics.silver.characters_state
   WHERE name_type = 'observed_current'
     AND is_active_lock = TRUE
),
filtered_identity AS (
  SELECT character_id,
         deleted_at
    FROM tibia_analytics.silver.characters_identity
)
SELECT /*+ BROADCAST(fs, fi) */
       ch.snapshot_id,
       fs.character_id,
       ch.name,
       ch.traded,
       fi.deleted_at AS deletion_at,
       ch.former_names,
       ch.title,
       ch.unlocked_titles,
       ch.sex,
       ch.vocation,
       ch.level,
       ch.achievement_points,
       ch.world_id,
       ch.world,
       ch.former_worlds,
       ch.residence,
       ch.married_to,
       ch.houses,
       ch.guild,
       ch.last_login_at,
       ch.comment,
       ch.position,
       ch.account_status,
       ch.account_badges,
       ch.achievements,
       ch.deaths_truncated,
       ch.deaths,
       ch.account_information,
       ch.other_characters,
       ch.source_file_date,
       ch.api_timestamp,
       current_timestamp() AS created_at,
       current_timestamp() AS processed_at
  FROM tibia_analytics.silver.characters_history AS ch
  LEFT JOIN filtered_state AS fs
    ON fs.name         = ch.name
   AND ch.source_file_date BETWEEN fs.valid_from AND fs.valid_to
  LEFT JOIN filtered_identity AS fi
    ON fi.character_id = fs.character_id;

In [0]:
-- Fix null character_id for character 'Infyy' to prevent identity pipeline failures
UPDATE tibia_analytics.silver.tmp_characters_history_enriched
   SET character_id     = '2db65c6a3d0d8c806d245740bceda2c073b0ad71'
 WHERE snapshot_id      = '6264b4d922f0bbd125ca3b3a7e6f4463590bcb8a'
   AND name             = 'Infyy'
   AND source_file_date = '2026-03-18'
   AND character_id IS NULL;

-- Fix null character_id for character 'Fizoleczek' to prevent identity pipeline failures
UPDATE tibia_analytics.silver.tmp_characters_history_enriched
   SET character_id     = '3f1f9d2d7d5682a26ca320e306cd20c1dc398657'
 WHERE snapshot_id      = 'c782c1c1f289c1813665e4c4d3e6bcddb1f62594'
   AND name             = 'Fizoleczek'
   AND source_file_date = '2026-03-04'
   AND character_id IS NULL;

-- Fix null character_id for character 'Naraiya Delfonar' to prevent identity pipeline failures
UPDATE tibia_analytics.silver.tmp_characters_history_enriched
   SET character_id     = '3ab87885ea611d5f40143aa85e13d8259ddc2063'
 WHERE snapshot_id      = '7505c75e37b1442fcb7a10b915e5cc4979acad40'
   AND name             = 'Naraiya Delfonar'
   AND source_file_date = '2026-04-08'
   AND character_id IS NULL;

In [0]:
CREATE OR REPLACE TABLE tibia_analytics.silver.tmp_characters_history_enriched_qualified AS
 SELECT *
   FROM tibia_analytics.silver.tmp_characters_history_enriched
QUALIFY ROW_NUMBER() OVER (
          PARTITION BY character_id, source_file_date
              ORDER BY api_timestamp DESC
        ) = 1;

In [0]:
MERGE INTO tibia_analytics.silver.characters_history_enriched AS target
USING tibia_analytics.silver.tmp_characters_history_enriched_qualified AS source
   ON target.snapshot_id      = source.snapshot_id
  AND target.character_id     = source.character_id
  AND target.source_file_date = source.source_file_date
WHEN NOT MATCHED THEN INSERT *;

In [0]:
DROP TABLE IF EXISTS tibia_analytics.silver.tmp_characters_history_enriched;
DROP TABLE IF EXISTS tibia_analytics.silver.tmp_characters_history_enriched_qualified;